# ETF Premium/Discount Mean Reversion — Research Notebook

Fixed income ETFs (HYG, LQD, JNK, TLT, AGG) trade continuously while their underlying bonds trade over the counter, so the ETF price can deviate from end-of-day NAV — especially in stress. This notebook runs the full research pipeline: data, spread construction, event study, regime analysis and backtest.

## 1. Motivation

**Hypothesis:** when the premium/discount z-score (63-day window) exceeds ±2σ, the dislocation reflects temporary liquidity pressure and mean-reverts over the following 1–20 trading days — faster after deep discounts and in high-VIX regimes.

In [ ]:
import logging
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 110

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'fetch.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # running from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

## 2. Data

`data/fetch.py` downloads 5 years of daily OHLCV for the five ETFs and the VIX via **yfinance**, credit OAS spreads (BAMLH0A0HYM2, BAMLC0A0CM) from **FRED**, and builds an end-of-day **NAV** series. FRED publishes no ETF NAV data, so NAV is simulated as price adjusted by a mean-reverting noise term bounded at ±0.5% (clearly logged; drop a real NAV export at `data/raw/<TICKER>_nav_external.csv` to override).

In [ ]:
from data import fetch

fetch.main()  # idempotent: re-downloads and overwrites data/raw/

hyg = pd.read_csv(fetch.RAW_DIR / 'HYG.csv', index_col=0, parse_dates=True)
hyg.tail()

In [ ]:
oas = pd.read_csv(fetch.RAW_DIR / 'oas.csv', index_col=0, parse_dates=True)
ax = oas.plot(figsize=(10, 3.5), title='Credit OAS spreads (FRED)')
ax.set_ylabel('OAS (%)')
plt.show()

## 3. Spread construction

For each ETF we compute `premium_discount = (Price − NAV) / NAV × 100`, its 63-day rolling z-score, and flag events where the z-score *crosses* ±2σ. Stress episodes (COVID-19 in Mar 2020, the Oct 2022 rate shock) are highlighted.

In [ ]:
from analysis import spread

spreads, zscores, events = spread.build_spread_panel()
spread.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
spreads.to_csv(spread.PROCESSED_DIR / 'premium_discount.csv')
zscores.to_csv(spread.PROCESSED_DIR / 'zscore.csv')
events.to_csv(spread.PROCESSED_DIR / 'events.csv', index=False)

fig = spread.plot_spread_history(spreads, events,
                                 spread.OUTPUTS_DIR / 'spread_history.png')
plt.show()
events.groupby(['ticker', 'side']).size().unstack(fill_value=0)

## 4. Event study

For every ±2σ crossing we compute forward ETF price returns at 1, 3, 5, 10 and 20 trading days, and the full 0–20 day cumulative return path with 95% confidence bands — separately for **discount** events (z < −2) and **premium** events (z > +2).

In [ ]:
from analysis import reversion

event_returns, summaries = reversion.run_event_study()
event_returns.to_csv(reversion.PROCESSED_DIR / 'event_study.csv', index=False)

fig = reversion.plot_reversion_paths(
    summaries, reversion.OUTPUTS_DIR / 'reversion_paths.png')
plt.show()

event_returns.groupby('side')[[f'ret_{h}d' for h in reversion.HORIZONS]].mean().round(3)

## 5. Regime analysis

Events are split by the VIX level on the event date: **high VIX** (> 20) vs **low VIX** (≤ 20). We compare both the *magnitude* of the subsequent reversion (average return path) and its *speed* (days for the z-score to revert to zero, censored at 20).

In [ ]:
from analysis import regime

tagged_events, regime_table = regime.run_regime_analysis()
regime_table.to_csv(regime.PROCESSED_DIR / 'regime_summary.csv')

closes = {t: spread.load_close(t) for t in fetch.ETF_TICKERS}
discount = tagged_events[tagged_events['side'] == 'discount']
summaries_by_regime = {
    r: reversion.path_summary(reversion.reversion_paths(
        discount[discount['regime'] == r], closes))
    for r in ('high_vix', 'low_vix')
    if not discount[discount['regime'] == r].empty
}
speed = discount.groupby('regime')[['days_to_revert']].mean()
fig = regime.plot_regime_comparison(
    summaries_by_regime, speed, regime.OUTPUTS_DIR / 'regime_comparison.png')
plt.show()
regime_table

## 6. Backtest

**Rules:** enter long when the discount z-score crosses below −2; exit when it crosses back through 0. Position size is the inverse of 21-day realised volatility, normalised and capped at 2×; signals execute at the next close (no look-ahead). Benchmark: buy-and-hold each ETF.

In [ ]:
from backtest import strategy

summary, results = strategy.run_backtests()
summary.to_csv(strategy.PROCESSED_DIR / 'backtest_summary.csv')

fig = strategy.plot_cumulative_returns(
    results, strategy.OUTPUTS_DIR / 'backtest_cumulative.png')
plt.show()
summary

## 7. Conclusion

* ±2σ premium/discount dislocations in fixed income ETFs are infrequent but cluster in liquidity stress (Mar 2020, Oct 2022).
* Discount events show positive average forward returns over 1–20 days, consistent with mean reversion; premium events show the mirror-image (weaker) pattern.
* Reversion after discount events tends to be **larger** in high-VIX regimes (deeper dislocations to revert from) though confidence bands are wide given few independent episodes.
* The simple long-discount strategy is in the market only a small fraction of the time and earns most of its return in stress windows; results are sensitive to the NAV data quality — replace the simulated NAV with issuer NAV files for production-grade conclusions.

**Caveats:** simulated NAV (unless overridden), no transaction costs/borrow, overlapping events inflate the effective sample, and the ±2σ threshold was not tuned out of sample.